In [1]:
import numpy as np 
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.colors import Normalize
import matplotlib.patches as mpatches

%config Inline.Backend.figure_format = 'png2x'

In [2]:
def load_outputs(outdir, vec_size=(2,200), y_size=(2,200)):
    """Load THETA and Y from one directory."""
    THETA = np.zeros((vec_size))
    Y = np.zeros((y_size))
    SIG = np.zeros((y_size))
    SIG_INT = np.zeros((y_size))

    # Load theta files
    for f in sorted(glob.glob(os.path.join(outdir, 'theta*tsv'))):
        theta = np.array(np.loadtxt(f), dtype=float).T
        THETA[:, :] = theta

    # Load Y files
    for f in sorted(glob.glob(os.path.join(outdir, 'Y*npy'))):
        y = np.load(f)
        Y[:, :] = y

    for f in sorted(glob.glob(os.path.join(outdir, 'SIG*npy'))):
        sig = np.load(f)
        SIG[:, :] = sig

    return THETA, Y, SIG

d = '/scratch/groups/earlew/arlenlex/P1-model-dev/nares/baseline_correct_geometry/state'

THETA, Y, SIG = load_outputs(d)

In [3]:
# --- indices for parameters in THETA ---
KB_IDX  = 0
EPS_IDX = 1  

# --- pull arrays and build masks ---
# Y has shape (N, 2, 10): Y[:,0,:] = peak1, Y[:,1,:] = peak2
y1 = Y[0, :].astype(float)
y2 = Y[1, :].astype(float)

is999_1 = y1 == 999
is999_2 = y2 == 999

# --- flatten everything so each column/run is a sample point ---
kb   = THETA[KB_IDX, :].astype(float).ravel()
eps  = THETA[EPS_IDX, :].astype(float).ravel()
m1   = is999_1.ravel()
m2   = is999_2.ravel()
y1_f = y1.ravel()
y2_f = y2.ravel()

# guard: keep only finite kb, eps (positive if you plan to log-scale)
valid_params = np.isfinite(kb) & np.isfinite(eps) & (kb > 0) & (eps > 0)
kb, eps, m1, m2, y1_f, y2_f = kb[valid_params], eps[valid_params], m1[valid_params], m2[valid_params], y1_f[valid_params], y2_f[valid_params]

# --- classify regimes ---
# 0 = no frac (999,999)
# 1 = stable arch (x,999)
# 2 = no arch (999,x)
# 3 = unstable arch (x1,x2)
regime = np.full(kb.shape, 3, dtype=int)        # default unstable arch
regime[m1 &  m2] = 0                            # (999, 999)
regime[~m1 & m2] = 1                            # (x, 999)
regime[m1 & ~m2] = 2                            # (999, x)
# (regime stays 3 when ~m1 & ~m2)

# --- collect values where any 999 appears ---
any_999 = m1 | m2
prod_kb_eps = kb * eps

def print_regime_values(tag, sel):
    if not np.any(sel):
        print(f"{tag}: none")
        return
    kb_sel, eps_sel, prod_sel = kb[sel], eps[sel], prod_kb_eps[sel]
    print(f"\n{tag}: count={sel.sum()}")
    print(f"  kb:   min={kb_sel.min():.3g}, max={kb_sel.max():.3g}")
    print(f"  eps:  min={eps_sel.min():.3g}, max={eps_sel.max():.3g}")
    print(f"  eps*kb: min={prod_sel.min():.3g}, max={prod_sel.max():.3g}")
    head = min(10, sel.sum())
    arr = np.column_stack([kb_sel[:head], eps_sel[:head], prod_sel[:head]])
    print("  first rows [kb, eps, eps*kb]:")
    with np.printoptions(precision=6, suppress=True):
        print(arr)

# Print: all with any 999, and split by regime
# print_regime_values("Any 999 (either peak)", any_999)
# print_regime_values("no frac (999,999)", regime == 0)
# print_regime_values("no arch (999,x)",   regime == 2)
# print_regime_values("stable arch (x,999)", regime == 1)

# --- colors: fixed for regimes 0–2; gradient for regime 3 by Δt between peaks ---
labels = {
    0: "no fracture",
    2: "no arch",
    1: "stable arch",
    3: "unstable arch",
}

# Base colors for regimes 0–2
fixed_cmap = ListedColormap(["pink", "r", "b", "g"])  # 'b' placeholder; regime 3 will be overridden

# Prepare RGBA array
colors = np.zeros((kb.size, 4))
colors[regime == 0] = fixed_cmap(0)   
colors[regime == 2] = fixed_cmap(1)   
colors[regime == 1] = fixed_cmap(2)  

# Regime 3: color by Δt = |y2 - y1|
reg3 = (regime == 3)
if np.any(reg3):
    dt = np.abs(y2_f[reg3] - y1_f[reg3])  # time between peaks
    # Normalize dt to [0,1] and map with 'Blues' (low -> light, high -> dark)
    norm = Normalize(vmin=np.nanmin(dt), vmax=np.nanmax(dt))
    blues = plt.get_cmap('Greens')
    colors_reg3 = blues(norm(dt))
    colors[reg3] = colors_reg3
else:
    norm = None  # no colorbar if no regime-3 points

In [4]:
# --- plot kb vs eps colored by regime ---
fig, ax = plt.subplots(figsize=(8,5))
# for i in range(len(THETA[0])):
#     plt.text(THETA[0, i], THETA[1, i], f'{i+1}', fontsize = 'small')
sc = plt.scatter(kb, eps, c=colors, s=66, edgecolor='k', linewidth=0.2)
plt.xscale('log'); plt.yscale('log')
plt.xlabel(r"$k_b$")
plt.ylabel(r"$\epsilon$")
plt.title("Strait Fracture Regimes")

# Legend for categorical regimes 0–2 + label for regime 3
handles = [
    mpatches.Patch(color=fixed_cmap(0), label=labels[0]),
    mpatches.Patch(color=fixed_cmap(1), label=labels[2]),
    mpatches.Patch(color=fixed_cmap(2), label=labels[1]),
    mpatches.Patch(color=fixed_cmap(3), label=labels[3])
]
plt.legend(handles=handles, frameon=False, fontsize='small', loc=(1.25, 0.45))

# Colorbar for regime 3 Δt
if norm is not None:
    sm = plt.cm.ScalarMappable(norm=norm, cmap='Greens')
    cbar = plt.colorbar(sm, ax =ax, pad=0.02)
    cbar.set_label(r"$\Delta t$ between peaks [s]", rotation=90)

plt.grid(True, ls=':', alpha=0.6)
plt.tight_layout()
plt.show()


ValueError: Data has no positive values, and therefore cannot be log-scaled.

Error in callback <function _draw_all_if_interactive at 0x7fc624eeaca0> (for post_execute), with arguments args (),kwargs {}:


ValueError: Data has no positive values, and therefore cannot be log-scaled.

ValueError: Data has no positive values, and therefore cannot be log-scaled.

<Figure size 800x500 with 1 Axes>